In [1]:
import arcpy

In [10]:
fc = r"D:\Documents\CityEngine\Default Workspace\Area_estudio\models\estudio.gdb\candidatos_cam"
fc_salida = r"D:\Documents\CityEngine\Default Workspace\Area_estudio\models\estudio.gdb\candidatos_cam_final"

In [3]:
puntos_max_z = {}

In [4]:
with arcpy.da.SearchCursor(fc, ["OID@", "SHAPE@X", "SHAPE@Y", "SHAPE@Z"]) as cursor:
    for oid, x, y, z in cursor:
        key = (round(x, 3), round(y, 3))

        if key not in puntos_max_z:
            puntos_max_z[key] = (oid, z)
        else:
            oid_guardado, z_guardado = puntos_max_z[key]
            if z > z_guardado:
                puntos_max_z[key] = (oid, z)

In [5]:
oids_validos = {oid for oid, z in puntos_max_z.values()}
oids_a_borrar = []

In [6]:
with arcpy.da.SearchCursor(fc, ["OID@"]) as cursor:
    for (oid,) in cursor:
        if oid not in oids_validos:
            oids_a_borrar.append(oid)

In [8]:
len(oids_a_borrar)

41917

In [11]:
desc = arcpy.Describe(fc)
path = desc.path
geom_type = desc.shapeType
spatial_ref = desc.spatialReference


In [12]:
arcpy.management.CreateFeatureclass(
    out_path=path,
    out_name=arcpy.Describe(fc_salida).name if "\\" not in fc_salida else fc_salida.split("\\")[-1],
    geometry_type=geom_type,
    spatial_reference=spatial_ref,
    has_m="DISABLED",
    has_z="ENABLED"
)

<Result 'D:\\Documents\\CityEngine\\Default Workspace\\Area_estudio\\models\\estudio.gdb\\candidatos_cam_final'>

In [13]:
campos_origen = [f.name for f in arcpy.ListFields(fc) 
                 if f.type not in ("OID", "Geometry")]

In [14]:
for campo in campos_origen:
    field = arcpy.ListFields(fc, campo)[0]
    arcpy.AddField_management(
        in_table=fc_salida,
        field_name=field.name,
        field_type=field.type,
        field_precision=field.precision,
        field_scale=field.scale,
        field_length=field.length,
        field_alias=field.aliasName,
        field_is_nullable=field.isNullable,
        field_is_required=field.required
    )


In [15]:
campos_cursor = ["SHAPE@"] + campos_origen

In [16]:
with arcpy.da.SearchCursor(fc, ["OID@", "SHAPE@"] + campos_origen) as cursor:
    with arcpy.da.InsertCursor(fc_salida, campos_cursor) as inserter:
        for row in cursor:
            oid = row[0]
            shape = row[1]
            atributos = row[2:]
            if oid in oids_validos:
                inserter.insertRow([shape] + list(atributos))